# 07 — News-Query (ohne Journalist:innen)

Liest [`data/accounts.csv`](../data/accounts.csv) und baut **eine** Brandwatch-Query
für alle News-Einträge, die **keine Journalist:innen** sind (also Zeitungen,
Rundfunksender, Nachrichtenprogramme, Online-Only, Nachrichtenagenturen,
Entertainment).

**Filter:**
- `category == "News"`
- `label != "Journalist"`
- `channel ∈ {x, instagram, facebook}`

**Struktur:** pro `label` ein OR-Block, Labels alphabetisch, Handles alphabetisch.
Platzsparend — Handles pro Block auf einer Zeile, ein `\n` pro Block.

**Output:** `output/queries/news_query.txt`.

⚠️ Bei ~3300 Handles kann die Query am 100k-Zeichen-Limit kratzen — unten wird
die Größe geloggt und ggf. gewarnt.

In [1]:
import os

import pandas as pd

if os.path.basename(os.getcwd()) == "scripts":
    PROJECT_ROOT = os.path.dirname(os.getcwd())
else:
    PROJECT_ROOT = os.getcwd()

DATA_DIR     = os.path.join(PROJECT_ROOT, "data")
QUERIES_DIR  = os.path.join(PROJECT_ROOT, "output", "queries")
ACCOUNTS_CSV = os.path.join(DATA_DIR, "accounts.csv")
OUTPUT_FILE  = os.path.join(QUERIES_DIR, "news_query.txt")

ALLOWED_CHANNELS = ["x", "instagram", "facebook"]
EXCLUDED_LABELS  = {"Journalist"}

LANGUAGE_FILTER = "language:de"

os.makedirs(QUERIES_DIR, exist_ok=True)

## 1. Daten laden + filtern

In [2]:
accounts = pd.read_csv(ACCOUNTS_CSV)

news = (
    accounts[
        (accounts["category"] == "News")
        & accounts["channel"].isin(ALLOWED_CHANNELS)
        & ~accounts["label"].isin(EXCLUDED_LABELS)
    ]
    .dropna(subset=["handle", "label"])
    .drop_duplicates(subset=["channel", "handle"])
    .copy()
)

print(f"Total nach Filter: {len(news)}")
print()
print("Label-Verteilung:")
print(news["label"].value_counts())

Total nach Filter: 3297

Label-Verteilung:
label
Rundfunksender         1298
Zeitung                 998
Online_Only             543
Entertainment           331
Nachrichtenprogramm      97
Nachrichtenagentur       30
Name: count, dtype: int64


## 2. Helper

In [3]:
def bw_author(handle: str) -> str:
    h = str(handle).strip().replace('"', '\\"')
    return f'author:"{h}"'


def block(comment: str, handles) -> str:
    body = " OR ".join(bw_author(h) for h in handles)
    return f"<<< {comment} — {len(handles)} Handles >>>\n({body})"

## 3. Blöcke pro Label (alphabetisch)

In [4]:
blocks: list[str] = []

for label in sorted(news["label"].unique(), key=str.lower):
    handles = (
        news[news["label"] == label]["handle"]
        .sort_values(key=lambda s: s.str.lower())
        .tolist()
    )
    blocks.append(block(label, handles))
    print(f"{label:<25s} {len(handles):>5} Handles")

print(f"\nBlöcke gesamt: {len(blocks)}")

Entertainment               331 Handles
Nachrichtenagentur           30 Handles
Nachrichtenprogramm          97 Handles
Online_Only                 543 Handles
Rundfunksender             1298 Handles
Zeitung                     998 Handles

Blöcke gesamt: 6


## 4. Gesamt-Query zusammensetzen

In [5]:
total_handles = sum(b.count('author:"') for b in blocks)
header = f"<<< News (ohne Journalist:innen) — {total_handles} Handles in {len(blocks)} Labels >>>"

body = "\nOR\n".join(blocks)
query = f"{header}\n({LANGUAGE_FILTER} AND (\n{body}\n))\n"

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    f.write(query)

size = os.path.getsize(OUTPUT_FILE)
print(f"Datei:           {OUTPUT_FILE}")
print(f"Größe:           {size:,} bytes  ({size / 1024:.1f} KiB)")
print(f"Handles gesamt:  {total_handles}")
print(f"Blöcke:          {len(blocks)}")

if size > 100_000:
    print(f"\n⚠️  {size - 100_000:,} Zeichen über dem 100k-Limit.")

Datei:           /Users/zorbeyozcan/Projekte/query_printer/output/queries/news_query.txt
Größe:           84,975 bytes  (83.0 KiB)
Handles gesamt:  3297
Blöcke:          6


## 5. Preview

In [6]:
with open(OUTPUT_FILE, "r", encoding="utf-8") as f:
    content = f.read()

for line in content.split("\n"):
    if "<<<" in line and ">>>" in line:
        print(line)

<<< News (ohne Journalist:innen) — 3297 Handles in 6 Labels >>>
<<< Entertainment — 331 Handles >>>
<<< Nachrichtenagentur — 30 Handles >>>
<<< Nachrichtenprogramm — 97 Handles >>>
<<< Online_Only — 543 Handles >>>
<<< Rundfunksender — 1298 Handles >>>
<<< Zeitung — 998 Handles >>>
